
## Store the raw data into bronze

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp


def load_to_bronze(
    spark: SparkSession,
    file_path: str,
    catalog_name: str,
    schema_name: str,
    bronze_table_name: str,
    drop_cols_name:list
) -> None:
    """
    Read a source Excel file and overwrite it into the bronze layer.
"""
    df = (
        spark.read
        .format("excel")
        .option("headerRows", 1)
        .load(file_path)
    )
    
    df = df.drop(*drop_cols_name)

    df_with_ts = df.withColumn("bronze_timestamp", current_timestamp())

    df_with_ts.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog_name}.{schema_name}.{bronze_table_name}")

In [0]:
drop_cols = ['source_system','operation_type']
file_path = "/Volumes/ct_oil_gas/sc_bronze/v1/raw_data/supply_chain_dataset.xlsx"
catalog_name = "ct_oil_gas"
schema_name = "sc_bronze"
bronze_table_name = "oil_gas_transactions"

load_to_bronze(spark, file_path, catalog_name, schema_name, bronze_table_name,drop_cols)

In [0]:
# md =spark.sql("SELECT * FROM ct_oil_gas.sc_metadata.ingestion_control Where active_flag='true' ORDER BY load_order")
# md.display()

In [0]:
# from typing import List
# from pyspark.sql import SparkSession, DataFrame, Row
# def load_metadata(spark: SparkSession) -> DataFrame:
#     """
#     Load active ingestion control metadata, ordered by load_order.
#     """
#     return spark.sql("""
#         SELECT * FROM ct_oil_gas.sc_metadata.ingestion_control
#         WHERE active_flag = 'true'
#         ORDER BY load_order
#     """)

# meta_data_df = load_metadata(spark)

# config_rows = meta_data_df.select(
#     'table_name',
#     'source_path',
#     'target_layer',
#     'target_catalog',
#     'bronze_schema',
#     'silver_schema',
#     'gold_schema',
#     'watermark_column',
# ).collect()

# for row in config_rows:
#     table_name = row["table_name"]
#     source_path = row["source_path"]
#     target_layer = row["target_layer"]
#     catalog_name = row["target_catalog"]
#     bronze_schema = row["bronze_schema"]
#     silver_schema = row["silver_schema"]
#     gold_schema = row["gold_schema"]

#     # process this table's ingestion here
#     print(f"Loading {table_name} from {source_path} into {catalog_name}.{bronze_schema}")